# Multi-Modal Genomic Transformer Model

This notebook implements a Transformer-based model designed for multi-task learning on genomic data. The model simultaneously predicts masked DNA nucleotides (a self-supervised task similar to Masked Language Modeling) and masked RNA expression states (a regression/classification task). This approach aims to leverage the interdependencies between DNA sequence and gene expression.

## 1. Imports and Global Constants

This section imports all necessary Python libraries and defines global constants used throughout the model. These constants include hyperparameters for the Transformer architecture, training parameters, and crucial values for the custom masking strategies.

In [ ]:
import torch
import torch.nn as nn
import math
import copy
import random
import os
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, random_split, Dataset, Subset
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt

# --- NEW GLOBAL CONSTANTS FOR MASKING ---
# Original number of nucleotide types (A, C, G, T, N)
NUM_NUCLEOTIDES_ORIGINAL = 5

# Index used for the MASK token in the one-hot encoded DNA input.
# If original nucleotides are 0-4, MASK will be 5.
MASK_NUCLEOTIDE_INDEX = NUM_NUCLEOTIDES_ORIGINAL

# One-hot dimension including original nucleotides + MASK token
ONE_HOT_NUCLEOTIDE_DIM = NUM_NUCLEOTIDES_ORIGINAL + 1 # Total one-hot dimension (A,C,G,T,N,MASK = 6)

# Special value for masked RNA expression in the input (distinct from 0.0 or 1.0)
MASK_EXPRESSION_VALUE = 0.5

# Masking probabilities
DNA_MASK_PROBABILITY = 0.15
RNA_MASK_FRACTION = 0.50

# BERT-like masking strategy probabilities for DNA
BERT_MASK_REPLACEMENT_PROB = 0.8  # 80% of the time: replace with MASK_TOKEN
BERT_RANDOM_REPLACEMENT_PROB = 0.1 # 10% of the time: replace with random original nucleotide
BERT_ORIGINAL_REPLACEMENT_PROB = 0.1 # 10% of the time: keep original nucleotide

# Model Hyperparameters
# Total input features per position: ONE_HOT_NUCLEOTIDE_DIM (6) + 1 (for Expression) = 7
NEW_INPUT_FEATURES = ONE_HOT_NUCLEOTIDE_DIM + 1

D_MODEL = 128       # Dimension of the model's embeddings
NUM_HEADS = 4       # Number of attention heads
NUM_LAYERS = 3      # Number of Transformer encoder layers
D_FF = 256          # Dimension of the feed-forward network
DROPOUT = 0.1       # Dropout rate

# Training Hyperparameters
BATCH_SIZE = 16
LEARNING_RATE = 0.0001
NUM_EPOCHS = 50
GRADIENT_ACCUMULATION_STEPS = 4
WARMUP_STEPS = 2000 # For learning rate scheduler

NUM_WORKERS = 2 # Number of data loading workers

# Placeholder for dynamically detected sequence length (window_size from data)
SEQ_LENGTH = None

## 2. Utility Functions

This section includes helper functions that are not part of the core model but are essential for the overall pipeline, such as determining the correct data directory based on the execution environment.

In [ ]:
# --- UTILITY FUNCTIONS ---

def get_data_dir():
    """
    Determines the correct data directory based on the execution environment.
    This function makes the code portable between Colab, local PyCharm, and cluster.
    """
    # Default to current working directory for local/cluster
    data_directory = os.path.join(os.getcwd(), 'data/')

    # Attempt to detect Google Colab and use Google Drive path
    try:
        from google.colab import drive
        drive.mount('/content/gdrive')
        # This path assumes your 'DnARnAProject' folder is directly in 'My Drive'
        google_drive_project_path = '/content/gdrive/MyDrive/DnARnAProject/'
        data_directory = os.path.join(google_drive_project_path, 'data/')
        print("Detected Google Colab environment. Using Google Drive path.")
    except ImportError:
        print("Not in Google Colab. Using local/cluster path.")

    if not os.path.isdir(data_directory):
        print(f"Error: The data directory '{data_directory}' does not exist.")
        print("Please ensure your data is located correctly (e.g., in a 'data/' folder relative to your script, or in Google Drive).")
        exit() # Exit if data directory is not found
    return data_directory

## 3. Transformer Core Modules

This section defines the fundamental building blocks of the Transformer architecture:
- **`MultiHeadAttention`**: Implements the core attention mechanism, allowing the model to weigh the importance of different parts of the input sequence.
- **`PositionWiseFeedForward`**: A simple two-layer feed-forward network applied independently to each position.
- **`PositionalEncoding`**: Adds information about the position of tokens in the sequence, as Transformers are permutation-invariant.

In [ ]:
# --- MODEL MODULES ---

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            # Fill masked positions with a very small number for softmax
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_probs = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_probs, V)
        return output, attn_probs

    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        attn_output, attn_probs = self.scaled_dot_product_attention(Q, K, V, mask)

        output = self.W_o(self.combine_heads(attn_output))
        return output, attn_probs


class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

## 4. Transformer Encoder Layer and Main Model

This section defines the full encoder layer of the Transformer and the main `DNASequenceClassifier` model. The `DNASequenceClassifier` is equipped with two distinct prediction heads for multi-task learning:
- One head for reconstructing masked DNA nucleotides.
- Another head for predicting masked RNA expression states.

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output, attn_probs = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output)) # Add & Norm for attention
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output)) # Add & Norm for feed-forward
        return x, attn_probs


class DNASequenceClassifier(nn.Module):
    def __init__(self, input_features, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout, num_nucleotides_original):
        """
        Initializes the DNASequenceClassifier for a masked language modeling (DNA)
        and masked expression prediction (RNA) task.

        Args:
            input_features (int): Number of input features per position.
                                  (ONE_HOT_NUCLEOTIDE_DIM + 1 for expression state)
            d_model (int): Dimension of the model's embeddings.
            num_heads (int): Number of attention heads.
            num_layers (int): Number of encoder layers.
            d_ff (int): Dimension of the feed-forward network.
            max_seq_length (int): Maximum sequence length for positional encoding.
            dropout (float): Dropout rate.
            num_nucleotides_original (int): Number of original nucleotide types (e.g., 5 for A,C,G,T,N).
                                           This is used for the DNA prediction head output size.
        """
        super(DNASequenceClassifier, self).__init__()

        self.input_projection = nn.Linear(input_features, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

        self.encoder_layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        # Prediction head for DNA nucleotide reconstruction (e.g., 5 logits for A,C,G,T,N)
        self.dna_prediction_head = nn.Linear(d_model, num_nucleotides_original)
        # Prediction head for RNA expression state prediction (1 logit for binary 0.0 or 1.0)
        self.expression_prediction_head = nn.Linear(d_model, 1)

    def forward(self, src):
        """
        Forward pass for the DNA Sequence Classifier in masked multi-task mode.

        Args:
            src (torch.Tensor): Input sequence tensor.
                                Expected shape: (batch_size, sequence_length, input_features)
        Returns:
            tuple:
                - dna_logits (torch.Tensor): Logits for DNA base prediction (shape: batch_size, seq_length, num_nucleotides_original)
                - expression_logits (torch.Tensor): Logits for expression prediction (shape: batch_size, seq_length, 1)
        """
        src_mask = None # Assuming no explicit padding mask needed if all window_sizes are fixed to SEQ_LENGTH

        src_embedded = self.input_projection(src) * math.sqrt(self.d_model)
        src_embedded = self.dropout(self.positional_encoding(src_embedded))

        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            # EncoderLayer returns output and attention probabilities, we only need output here
            enc_output, _ = enc_layer(enc_output, src_mask)

        dna_logits = self.dna_prediction_head(enc_output)
        expression_logits = self.expression_prediction_head(enc_output)

        return dna_logits, expression_logits

## 5. Data Loading - Base Dataset (`GenomeExpressionDataset`)

This custom PyTorch `Dataset` handles the initial loading and preprocessing of genomic data. It reads DNA sequences and corresponding RNA expression labels from `.npz` and `.parquet` files. Key features include:
- Loading integer-encoded DNA sequences and float32 expression values.
- Applying reverse complement transformation for sequences on the '-' strand.
- Providing chromosome information, which is crucial for a robust train/validation/test split.

In [ ]:
# --- DATASET CLASSES ---

class GenomeExpressionDataset(Dataset):
    """
    Custom Dataset for loading DNA sequence and expression data for genomic regions.
    Loads data from pre-processed .npz and .parquet files.
    Handles reverse complement for '-' strand sequences.
    Returns integer-encoded DNA sequences and float32 expression labels.
    Provides chromosome information for splitting.
    """
    def __init__(self, data_dir):
        self.data_dir = data_dir

        self.data_npz_path = os.path.join(data_dir, 'data.npz')
        self.regions_parquet_path = os.path.join(data_dir, 'regions.parquet')

        try:
            loaded_npz = np.load(self.data_npz_path, allow_pickle=True)
            self.sequence_data = loaded_npz['sequence'] # Stored as integers (0-4)
            self.expression_plus_data = loaded_npz['expressed_plus']
            self.expression_minus_data = loaded_npz['expressed_minus']
            loaded_npz.close()
        except KeyError as e:
            available_keys = list(np.load(self.data_npz_path).keys()) if os.path.exists(self.data_npz_path) else "File not found during key check."
            raise RuntimeError(f"KeyError: Key '{e}' not found in {self.data_npz_path}. "
                               f"Available keys: {available_keys}. "
                               f"Please check your .npz file structure.")
        except Exception as e:
            raise RuntimeError(f"Could not load data from {self.data_npz_path}. Make sure the file exists and is not corrupted: {e}")

        try:
            self.regions_df = pd.read_parquet(self.regions_parquet_path)
            # Ensure 'chromosome' column exists
            if 'chromosome' not in self.regions_df.columns:
                raise ValueError(f"The 'regions.parquet' file must contain a 'chromosome' column for splitting.")
        except Exception as e:
            raise RuntimeError(f"Could not load regions from {self.regions_parquet_path}. Make sure the file exists and is not corrupted: {e}")

        assert len(self.sequence_data) == len(self.expression_plus_data) == len(self.expression_minus_data)
        assert len(self.sequence_data) == len(self.regions_df)

        # Map for reverse complement: A<->T, C<->G, N<->N (0<->3, 1<->2, 4<->4)
        self.complement_map = np.array([3, 2, 1, 0, 4], dtype=np.uint8)

    def __len__(self):
        return len(self.regions_df)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        region_info = self.regions_df.iloc[idx]

        offset = region_info['offset']
        window_size = region_info['window_size']
        strand = region_info['strand']

        # Get original sequence (integer encoded) and expression labels
        original_nucleotide_segment = self.sequence_data[offset : offset + window_size][::-1].copy()

        if strand == '+':
            original_expression_label_segment = self.expression_plus_data[offset : offset + window_size].copy()
        else: # strand == '-'
            # Apply reverse complement to DNA sequence
            original_nucleotide_segment = self.complement_map[original_nucleotide_segment][::-1].copy()
            # Corresponding expression label segment for the minus strand (already handled by preprocessing)
            original_expression_label_segment = self.expression_minus_data[offset : offset + window_size].copy()

        # Ensure expression_label_segment is a float32 tensor
        original_expression_labels = torch.tensor(original_expression_label_segment, dtype=torch.float32)

        # Return original integer sequence and expression labels
        return torch.tensor(original_nucleotide_segment, dtype=torch.long), original_expression_labels

    def get_all_sample_chromosomes(self):
        """Returns a list of chromosome identifiers for all samples."""
        return self.regions_df['chromosome'].tolist()

## 6. Data Loading - Masking Dataset (`MultiModalMaskingDataset`)

This is a crucial component that wraps the `GenomeExpressionDataset` to apply the specific masking strategies required for multi-task learning:
- **RNA Masking**: A *contiguous 50%* span of RNA expression values is masked with a special value (`MASK_EXPRESSION_VALUE`).
- **DNA Masking**: *Random 15%* of DNA nucleotides are masked using a BERT-like strategy (80% `[MASK]` token, 10% random nucleotide, 10% original nucleotide).

This dataset prepares the input for the model (`model_input`) and provides the corresponding ground truth labels (`original_nucleotide_labels`, `original_expression_labels`) and target masks (`dna_target_mask`, `rna_target_mask`) indicating which positions need to be predicted.

In [ ]:
class MultiModalMaskingDataset(Dataset):
    """
    Wraps GenomeExpressionDataset to apply both contiguous RNA masking (50%)
    and random DNA masking (15%) for a multi-task prediction.
    """
    def __init__(self, base_dataset: GenomeExpressionDataset, seq_length: int,
                 dna_mask_probability=DNA_MASK_PROBABILITY, rna_mask_fraction=RNA_MASK_FRACTION):
        self.base_dataset = base_dataset
        self.seq_length = seq_length # Fixed sequence length from global constant
        self.dna_mask_probability = dna_mask_probability
        self.rna_mask_fraction = rna_mask_fraction

        self.num_nucleotides_original = NUM_NUCLEOTIDES_ORIGINAL
        self.one_hot_nucleotide_dim = ONE_HOT_NUCLEOTIDE_DIM
        self.mask_nucleotide_index = MASK_NUCLEOTIDE_INDEX
        self.mask_expression_value = MASK_EXPRESSION_VALUE

        self.bert_mask_replacement_prob = BERT_MASK_REPLACEMENT_PROB
        self.bert_random_replacement_prob = BERT_RANDOM_REPLACEMENT_PROB
        self.bert_original_replacement_prob = BERT_ORIGINAL_REPLACEMENT_PROB

    def __len__(self):
        return len(self.base_dataset)

    def _one_hot_encode_dna(self, sequence_segment_int, one_hot_dim):
        """
        Helper to one-hot encode nucleotide integer indices for model input.
        Handles the expanded dimension for the MASK token.
        """
        one_hot_tensor = torch.zeros(len(sequence_segment_int), one_hot_dim, dtype=torch.float32)
        # Scatter original nucleotide values (0-4)
        one_hot_tensor.scatter_(1, sequence_segment_int.unsqueeze(1).long(), 1)
        return one_hot_tensor

    def __getitem__(self, idx):
        # Retrieve original integer-encoded DNA sequence and float32 expression labels
        # original_nucleotide_labels: (seq_len,) -> integer indices 0-4
        # original_expression_labels: (seq_len,) -> float32 values 0.0 or 1.0
        original_nucleotide_labels, original_expression_labels = self.base_dataset[idx]

        # Initialize tensors for model input and target masks
        # For model_input, we need to convert original_nucleotide_labels to one-hot
        input_nucleotide_features = self._one_hot_encode_dna(original_nucleotide_labels, self.one_hot_nucleotide_dim) # (seq_len, 6)

        # Input expression state will be (seq_len, 1)
        input_expression_states = original_expression_labels.clone().unsqueeze(-1) # (seq_len, 1)

        # Initialize masks for what we actually need to predict
        dna_target_mask = torch.full((self.seq_length,), False, dtype=torch.bool)
        rna_target_mask = torch.full((self.seq_length,), False, dtype=torch.bool)

        # --- Apply RNA Masking (Contiguous 50%) ---
        rna_mask_span_length = max(1, int(self.seq_length * self.rna_mask_fraction))
        if self.seq_length > rna_mask_span_length:
            rna_mask_start_idx = random.randint(0, self.seq_length - rna_mask_span_length)
        else: # Mask the entire sequence if span length is >= sequence length
            rna_mask_start_idx = 0
            rna_mask_span_length = self.seq_length
        rna_mask_end_idx = rna_mask_start_idx + rna_mask_span_length

        input_expression_states[rna_mask_start_idx : rna_mask_end_idx] = self.mask_expression_value
        rna_target_mask[rna_mask_start_idx : rna_mask_end_idx] = True

        # --- Apply DNA Masking (Random 15% with BERT-like strategy) ---
        for i in range(self.seq_length):
            if random.random() < self.dna_mask_probability:
                dna_target_mask[i] = True # Mark this position for DNA prediction

                rand_val = random.random()
                if rand_val < self.bert_mask_replacement_prob:
                    # 80% of the time: replace with MASK_NUCLEOTIDE_INDEX
                    input_nucleotide_features[i] = torch.zeros(self.one_hot_nucleotide_dim)
                    input_nucleotide_features[i, self.mask_nucleotide_index] = 1.0
                elif rand_val < (self.bert_mask_replacement_prob + self.bert_random_replacement_prob):
                    # 10% of the time: replace with a random original nucleotide (0-4)
                    random_nuc_idx = random.randint(0, self.num_nucleotides_original - 1)
                    input_nucleotide_features[i] = torch.zeros(self.one_hot_nucleotide_dim)
                    input_nucleotide_features[i, random_nuc_idx] = 1.0
                # else (10% of the time): keep original nucleotide (no change to input_nucleotide_features[i] needed)

        # Concatenate nucleotide features with expression states to form model input
        # Resulting shape: (seq_length, ONE_HOT_NUCLEOTIDE_DIM + 1)
        model_input = torch.cat((input_nucleotide_features, input_expression_states), dim=-1)

        # Return:
        # model_input: (seq_len, ONE_HOT_NUCLEOTIDE_DIM + 1) with masked values for model input
        # original_nucleotide_labels: (seq_len,) -> true integer labels (0-4) for DNA prediction target
        # original_expression_labels: (seq_len,) -> true float labels (0.0/1.0) for RNA prediction target
        # dna_target_mask: (seq_len,) -> boolean, True where DNA was masked and needs prediction
        # rna_target_mask: (seq_len,) -> boolean, True where RNA was masked and needs prediction
        return model_input, original_nucleotide_labels, original_expression_labels, dna_target_mask, rna_target_mask

## 7. Data Loading and Chromosome-Based Splitting Logic

This section orchestrates the data loading and prepares the datasets for training, validation, and testing. It dynamically determines the sequence length from the loaded data and implements a robust chromosome-based splitting strategy to ensure that training, validation, and test sets contain data from entirely separate chromosomes. This prevents data leakage and provides a more realistic evaluation of the model's generalization capabilities.

A `DEBUG_DATASET_SIZE` option is included to allow for faster testing with a smaller subset of the data.

In [ ]:
# --- DATA LOADING AND CHROMOSOME-BASED SPLITTING LOGIC ---

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Get data directory
data_dir = get_data_dir()

# Initialize base dataset
try:
    base_full_dataset = GenomeExpressionDataset(data_dir)
    print(f"Full base dataset loaded successfully. Total samples: {len(base_full_dataset)}")

    if len(base_full_dataset) > 0:
        # Dynamically determine SEQ_LENGTH (window_size) from the first sample
        sample_dna_int, _ = base_full_dataset[0]
        global SEQ_LENGTH # Ensure we modify the global variable
        SEQ_LENGTH = sample_dna_int.shape[0]
        print(f"Detected sequence length (window_size): {SEQ_LENGTH}")
    else:
        print("Warning: Base dataset is empty. Cannot determine SEQ_LENGTH dynamically. Please ensure data is present.")
        exit() # Exit if dataset is empty to prevent errors

except (FileNotFoundError, RuntimeError, ValueError) as e:
    print(f"Error loading base dataset: {e}")
    print("Please ensure your data files ('data.npz', 'regions.parquet') are correctly placed and formatted.")
    exit() # Exit execution if essential files/data are missing or malformed

# --- Debugging/Subset Training Configuration ---
DEBUG_DATASET_SIZE = None # Set to an integer (e.g., 200000) for a subset; Set to None for full dataset

dataset_to_split_indices = list(range(len(base_full_dataset)))
if DEBUG_DATASET_SIZE is not None and DEBUG_DATASET_SIZE < len(base_full_dataset):
    print(f"DEBUG MODE: Creating a subset of size {DEBUG_DATASET_SIZE} from the full dataset.")
    random.shuffle(dataset_to_split_indices) # Shuffle indices for a random subset
    dataset_to_split_indices = dataset_to_split_indices[:DEBUG_DATASET_SIZE]
    print(f"Subset created with {len(dataset_to_split_indices)} samples.")
else:
    print("Using the full dataset for splitting.")

# Create a Subset of the base_full_dataset containing only the samples to be split
subset_base_dataset = Subset(base_full_dataset, dataset_to_split_indices)

# Retrieve chromosome information for the *subset* of samples
# This ensures that chromosome counts and splits are based on the actual samples being used.
sample_chromosomes_for_subset = [base_full_dataset.regions_df.iloc[i]['chromosome'] for i in dataset_to_split_indices]

# Define unique chromosomes and counts from the subset of samples
all_chromosomes = sorted(list(set(sample_chromosomes_for_subset)))
num_chromosomes = len(all_chromosomes)

print(f"\nTotal unique chromosomes found in the selected samples: {num_chromosomes}")

# Define split ratios for chromosomes
train_chrom_ratio = 0.8
val_chrom_ratio = 0.1
test_chrom_ratio = 0.1 # Implicit, as it takes the rest

# Calculate chromosome counts for each split
train_chrom_count = max(1, int(num_chromosomes * train_chrom_ratio)) # Ensure at least 1 chromosome
val_chrom_count = max(0, int(num_chromosomes * val_chrom_ratio))
# Test count takes the remainder, ensuring no overlap and covering all chromosomes
test_chrom_count = max(0, num_chromosomes - train_chrom_count - val_chrom_count)

# Adjust counts if total chromosomes are too few for a 3-way split
if num_chromosomes < 3:
    print("Warning: Less than 3 unique chromosomes available for splitting. Adjusting strategy.")
    if num_chromosomes == 0:
        print("Error: No chromosomes found in data. Cannot proceed with splitting.")
        exit()
    elif num_chromosomes == 1:
        train_chroms = all_chromosomes
        val_chroms = []
        test_chroms = []
        print("Using single chromosome for training only (no separate validation/test sets).")
    else: # num_chromosomes == 2
        train_chroms = [all_chromosomes[0]]
        val_chroms = [all_chromosomes[1]]
        test_chroms = []
        print("Using one chromosome for training, one for validation (no separate test set).")
else: # Standard splitting for 3 or more chromosomes
    train_chroms = all_chromosomes[:train_chrom_count]
    val_chroms = all_chromosomes[train_chrom_count : train_chrom_count + val_chrom_count]
    test_chroms = all_chromosomes[train_chrom_count + val_chrom_count :]

# If validation or test sets are empty due to small number of chromosomes, ensure no index lookup errors
if not val_chroms and val_chrom_count > 0:
    print("Warning: Not enough chromosomes for validation set, it will be empty.")
if not test_chroms and test_chrom_count > 0:
    print("Warning: Not enough chromosomes for test set, it will be empty.")

print(f"\nChromosomes assigned to Training: {train_chroms} ({len(train_chroms)} chromosomes)")
print(f"Chromosomes assigned to Validation: {val_chroms} ({len(val_chroms)} chromosomes)")
print(f"Chromosomes assigned to Test: {test_chroms} ({len(test_chroms)} chromosomes)")


# Filter indices based on chromosome assignment
train_indices = [idx for idx, chrom in enumerate(sample_chromosomes_for_subset) if chrom in train_chroms]
val_indices = [idx for idx, chrom in enumerate(sample_chromosomes_for_subset) if chrom in val_chroms]
test_indices = [idx for idx, chrom in enumerate(sample_chromosomes_for_subset) if chrom in test_chroms]


# Create Subset datasets from the full original dataset using the generated indices
# This ensures that MultiModalMaskingDataset always receives an instance of GenomeExpressionDataset or its Subset.
train_dataset_raw = Subset(base_full_dataset, train_indices)
val_dataset_raw = Subset(base_full_dataset, val_indices)
test_dataset_raw = Subset(base_full_dataset, test_indices)

# Apply the MultiModalMaskingDataset wrapper to these raw subsets
train_dataset = MultiModalMaskingDataset(train_dataset_raw, seq_length=SEQ_LENGTH)
val_dataset = MultiModalMaskingDataset(val_dataset_raw, seq_length=SEQ_LENGTH)
test_dataset = MultiModalMaskingDataset(test_dataset_raw, seq_length=SEQ_LENGTH)


# Setup DataLoaders for each split
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"\nNumber of samples in training set: {len(train_dataset)}")
print(f"Number of samples in validation set: {len(val_dataset)}")
print(f"Number of samples in test set: {len(test_dataset)}")

## 8. Model, Loss, Optimizer Initialization & Metrics Setup

This section initializes the `DNASequenceClassifier` model and sets up the training components:
- **Model Instantiation**: Creates an instance of the Transformer model.
- **Loss Functions**: Defines `CrossEntropyLoss` for DNA nucleotide prediction and `BCEWithLogitsLoss` for binary RNA expression prediction. Both are configured to sum losses over masked tokens for batch-wise aggregation.
- **Optimizer**: Uses `Adam` for efficient gradient descent.
- **Mixed Precision Training**: `GradScaler` is utilized for Automatic Mixed Precision (AMP), which helps in faster training and reduced memory usage.
- **Learning Rate Scheduler**: A `LambdaLR` scheduler implements a warm-up phase for stable training at the beginning.
- **Metric Storage**: Lists are prepared to record training and validation losses and accuracies across epochs for later visualization.

In [ ]:
# --- MODEL INITIALIZATION ---
model = DNASequenceClassifier(
    input_features=NEW_INPUT_FEATURES,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    max_seq_length=SEQ_LENGTH, # Use the dynamically detected sequence length
    dropout=DROPOUT,
    num_nucleotides_original=NUM_NUCLEOTIDES_ORIGINAL
).to(device)

# Define two separate loss criteria for DNA and RNA
dna_criterion = nn.CrossEntropyLoss(reduction='sum') # Sum over masked positions for total loss
rna_criterion = nn.BCEWithLogitsLoss(reduction='sum') # Sum over masked positions for total loss

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.98), eps=1e-9)
scaler = GradScaler() # For Automatic Mixed Precision (AMP)

# Learning rate scheduler
def lr_lambda(current_step: int):
    if current_step < WARMUP_STEPS:
        return float(current_step) / float(max(1, WARMUP_STEPS))
    return 1.0

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# --- Lists to store metrics for plotting ---
train_total_losses = []
val_total_losses = []
train_dna_accuracies = []
val_dna_accuracies = []
train_rna_accuracies = []
val_rna_accuracies = []

## 9. Training Loop

This is the core training logic for the model. It iterates through epochs and batches, performing the following steps:
- **Forward Pass**: Passes the masked input sequences through the model to get DNA and RNA logits.
- **Loss Calculation**: Calculates separate losses for DNA nucleotide prediction (CrossEntropy) and RNA expression prediction (BCEWithLogits) using only the masked tokens.
- **Multi-task Loss**: Combines the DNA and RNA losses.
- **Gradient Accumulation**: Divides the total loss by `GRADIENT_ACCUMULATION_STEPS` and accumulates gradients over multiple batches before performing an optimizer step. This effectively increases the batch size without requiring more GPU memory.
- **Mixed Precision Training**: Utilizes `torch.cuda.amp.autocast` and `GradScaler` to train with mixed precision, speeding up computation and reducing memory footprint on compatible hardware.
- **Metrics Tracking**: Accumulates total loss, DNA accuracy, and RNA accuracy for both training and validation phases.

In [ ]:
print("\nStarting training...")

global_optimizer_step = 0  # To track total optimizer steps for LR scheduler

for epoch in range(NUM_EPOCHS):
    # --- Training Loop ---
    model.train()
    total_train_loss = 0.0
    total_dna_correct_predictions = 0
    total_dna_masked_tokens = 0
    total_rna_correct_predictions = 0
    total_rna_masked_tokens = 0

    if device.type == 'cuda':
        torch.cuda.empty_cache()

    for batch_idx, (sequences_input, original_nucleotide_labels, original_expression_labels, dna_target_masks, rna_target_masks) in enumerate(train_loader):
        # Move data to device
        sequences_input = sequences_input.to(device) # (batch_size, seq_len, NEW_INPUT_FEATURES)
        original_nucleotide_labels = original_nucleotide_labels.to(device) # (batch_size, seq_len) - integer indices
        # Unsqueeze RNA target to (batch_size, seq_len, 1) for consistent dimensions
        original_expression_labels = original_expression_labels.to(device)
        dna_target_masks = dna_target_masks.to(device) # (batch_size, seq_len) - bool
        rna_target_masks = rna_target_masks.to(device) # (batch_size, seq_len) - bool

        # Zero gradients at the beginning of accumulation steps
        if batch_idx % GRADIENT_ACCUMULATION_STEPS == 0:
            optimizer.zero_grad()

        with autocast(): # For mixed precision training
            # Model returns two sets of logits
            dna_logits, expression_logits = model(sequences_input) # (batch, seq_len, NUM_NUCLEOTIDES_ORIGINAL), (batch, seq_len)

            # --- Calculate DNA Loss ---
            # Flatten to (total_positions, num_nuc_original) for CrossEntropyLoss
            dna_logits_flat = dna_logits.view(-1, NUM_NUCLEOTIDES_ORIGINAL)
            original_nucleotide_labels_flat = original_nucleotide_labels.view(-1)
            dna_target_masks_flat = dna_target_masks.view(-1)

            masked_dna_logits = dna_logits_flat[dna_target_masks_flat]
            masked_original_nucleotide_labels = original_nucleotide_labels_flat[dna_target_masks_flat]

            dna_loss_batch = torch.tensor(0.0, device=device) # Initialize to avoid errors if no masked tokens
            if masked_dna_logits.numel() > 0:
                dna_loss_batch = dna_criterion(masked_dna_logits, masked_original_nucleotide_labels) # Sums over masked tokens

            # --- Calculate RNA Loss ---
            expression_logits_flat = expression_logits.view(-1) # Flatten to (total_positions,)
            original_expression_labels_flat = original_expression_labels.view(-1) # Flatten to (total_positions,)
            rna_target_masks_flat = rna_target_masks.view(-1)

            masked_expression_logits = expression_logits_flat[rna_target_masks_flat]
            masked_original_expression_labels = original_expression_labels_flat[rna_target_masks_flat]

            rna_loss_batch = torch.tensor(0.0, device=device) # Initialize to avoid errors if no masked tokens
            if masked_expression_logits.numel() > 0:
                rna_loss_batch = rna_criterion(masked_expression_logits, masked_original_expression_labels) # Sums over masked tokens

            # Combine losses
            total_batch_loss_sum = dna_loss_batch + rna_loss_batch
            # Scale loss for gradient accumulation (average loss over samples per accumulation step)
            loss = total_batch_loss_sum / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(loss).backward() # Scale loss and perform backward pass

        # Perform optimizer step and scheduler step after accumulation
        if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            global_optimizer_step += 1

        # --- Accumulate Metrics for Training ---
        if total_batch_loss_sum.numel() > 0:
             total_train_loss += total_batch_loss_sum.item()

        if masked_dna_logits.numel() > 0:
            dna_predictions = torch.argmax(masked_dna_logits, dim=-1)
            total_dna_correct_predictions += (dna_predictions == masked_original_nucleotide_labels).sum().item()
            total_dna_masked_tokens += masked_original_nucleotide_labels.size(0)

        if masked_expression_logits.numel() > 0:
            rna_predictions = (torch.sigmoid(masked_expression_logits) > 0.5).float()
            total_rna_correct_predictions += (rna_predictions == masked_original_expression_labels).sum().item()
            total_rna_masked_tokens += masked_original_expression_labels.size(0)

        # Print progress every 100 batches
        if (batch_idx + 1) % 100 == 0:
            avg_dna_acc_so_far = total_dna_correct_predictions / (total_dna_masked_tokens if total_dna_masked_tokens > 0 else 1)
            avg_rna_acc_so_far = total_rna_correct_predictions / (total_rna_masked_tokens if total_rna_masked_tokens > 0 else 1)
            # Normalize total loss by sum of masked samples so far
            avg_total_loss_so_far = total_train_loss / ((total_dna_masked_tokens + total_rna_masked_tokens) if (total_dna_masked_tokens + total_rna_masked_tokens) > 0 else 1)
            current_lr = optimizer.param_groups[0]['lr']
            print(f"  Batch {batch_idx + 1}/{len(train_loader)}, Current Loss (per sample in batch): {loss.item()*GRADIENT_ACCUMULATION_STEPS:.6f}, Accumulated Loss (total avg): {avg_total_loss_so_far:.6f}, "
                  f"DNA Acc: {avg_dna_acc_so_far:.4f}, RNA Acc: {avg_rna_acc_so_far:.4f}, LR: {current_lr:.6f}")

    # Final optimizer step for leftover batches that didn't complete an accumulation cycle
    if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad() # Clear gradients after final update
        scheduler.step()
        global_optimizer_step += 1

    # Calculate average epoch metrics
    avg_train_total_loss = total_train_loss / ((total_dna_masked_tokens + total_rna_masked_tokens) if (total_dna_masked_tokens + total_rna_masked_tokens) > 0 else 1)
    train_dna_accuracy = total_dna_correct_predictions / (total_dna_masked_tokens if total_dna_masked_tokens > 0 else 1)
    train_rna_accuracy = total_rna_correct_predictions / (total_rna_masked_tokens if total_rna_masked_tokens > 0 else 1)

    train_total_losses.append(avg_train_total_loss)
    train_dna_accuracies.append(train_dna_accuracy)
    train_rna_accuracies.append(train_rna_accuracy)

    # Print epoch summary in bold green
    BOLD = "\033[1m"
    GREEN = "\033[92m"
    RESET = "\033[0m"
    print(f"\n{BOLD}Epoch {epoch + 1}/{NUM_EPOCHS}:{RESET}")
    print(f"{BOLD}  Train Total Loss (avg over masked tokens): {GREEN}{avg_train_total_loss:.6f}{RESET}")
    print(f"{BOLD}  Train DNA Accuracy (masked): {GREEN}{train_dna_accuracy:.4f}{RESET}")
    print(f"{BOLD}  Train RNA Accuracy (masked): {GREEN}{train_rna_accuracy:.4f}{RESET}")

## 10. Validation Loop

After each training epoch, the model is evaluated on the validation set to monitor its performance on unseen data and detect overfitting. Key aspects:
- **Evaluation Mode**: `model.eval()` disables dropout and batch normalization updates.
- **No Gradient Calculation**: `torch.no_grad()` is used to prevent gradient calculations, saving memory and speeding up computation.
- **Metrics Collection**: Similar to training, validation losses and accuracies are calculated and stored.

In [ ]:
    # --- Validation Loop ---
    model.eval()
    val_total_loss = 0.0
    val_dna_correct_predictions = 0
    val_dna_masked_samples = 0
    val_rna_correct_predictions = 0
    val_rna_masked_samples = 0

    with torch.no_grad():
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        for sequences_input, original_nucleotide_labels, original_expression_labels, dna_target_masks, rna_target_masks in val_loader:
            sequences_input = sequences_input.to(device)
            original_nucleotide_labels = original_nucleotide_labels.to(device)
            original_expression_labels = original_expression_labels.to(device)
            dna_target_masks = dna_target_masks.to(device)
            rna_target_masks = rna_target_masks.to(device)

            with autocast():
                dna_logits, expression_logits = model(sequences_input)

                # --- Calculate DNA Loss ---
                dna_logits_flat = dna_logits.view(-1, NUM_NUCLEOTIDES_ORIGINAL)
                original_nucleotide_labels_flat = original_nucleotide_labels.view(-1)
                dna_target_masks_flat = dna_target_masks.view(-1)
                masked_dna_logits = dna_logits_flat[dna_target_masks_flat]
                masked_original_nucleotide_labels = original_nucleotide_labels_flat[dna_target_masks_flat]
                dna_loss_batch = torch.tensor(0.0, device=device)
                if masked_dna_logits.numel() > 0:
                    dna_loss_batch = dna_criterion(masked_dna_logits, masked_original_nucleotide_labels)

                # --- Calculate RNA Loss ---
                expression_logits_flat = expression_logits.view(-1)
                original_expression_labels_flat = original_expression_labels.view(-1)
                rna_target_masks_flat = rna_target_masks.view(-1)
                masked_expression_logits = expression_logits_flat[rna_target_masks_flat]
                masked_original_expression_labels = original_expression_labels_flat[rna_target_masks_flat]
                rna_loss_batch = torch.tensor(0.0, device=device)
                if masked_expression_logits.numel() > 0:
                    rna_loss_batch = rna_criterion(masked_expression_logits, masked_original_expression_labels)

                val_total_loss_sum = dna_loss_batch + rna_loss_batch
            val_total_loss += val_total_loss_sum.item()

            # --- Accumulate Metrics for Validation ---
            if masked_dna_logits.numel() > 0:
                dna_predictions = torch.argmax(masked_dna_logits, dim=-1)
                val_dna_correct_predictions += (dna_predictions == masked_original_nucleotide_labels).sum().item()
                val_dna_masked_samples += masked_original_nucleotide_labels.size(0)

            if masked_expression_logits.numel() > 0:
                rna_predictions = (torch.sigmoid(masked_expression_logits) > 0.5).float()
                val_rna_correct_predictions += (rna_predictions == masked_original_expression_labels).sum().item()
                val_rna_masked_samples += masked_original_expression_labels.size(0)

    # Calculate average validation metrics for the epoch
    avg_val_total_loss = val_total_loss / ((val_dna_masked_samples + val_rna_masked_samples) if (val_dna_masked_samples + val_rna_masked_samples) > 0 else 1)
    val_dna_accuracy = val_dna_correct_predictions / (val_dna_masked_samples if val_dna_masked_samples > 0 else 1)
    val_rna_accuracy = val_rna_correct_predictions / (val_rna_masked_samples if val_rna_masked_samples > 0 else 1)

    val_total_losses.append(avg_val_total_loss)
    val_dna_accuracies.append(val_dna_accuracy)
    val_rna_accuracies.append(val_rna_accuracy)

    print(f"  Validation Total Loss: {avg_val_total_loss:.4f}, Validation DNA Accuracy: {val_dna_accuracy:.4f}, Validation RNA Accuracy: {val_rna_accuracy:.4f}")

print("\nTraining complete!")

## 11. Test Evaluation

After training is complete, the model's final performance is assessed on the completely unseen test set. This provides an unbiased estimate of how well the model generalizes to new data. In addition to overall loss and accuracy, this section collects and stores the original and predicted masked DNA nucleotides for a detailed distribution analysis.

In [ ]:
# --- Test Evaluation ---
print("\nStarting test evaluation...")
model.eval()
test_total_loss = 0
test_dna_correct_predictions = 0
test_dna_masked_samples = 0
test_rna_correct_predictions = 0
test_rna_masked_samples = 0

# New lists to store original and predicted masked nucleotides for distribution comparison
all_original_masked_nucleotides = []
all_predicted_nucleotides = []

with torch.no_grad():
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    for sequences_input, original_nucleotide_labels, original_expression_labels, dna_target_masks, rna_target_masks in test_loader:
        sequences_input = sequences_input.to(device)
        original_nucleotide_labels = original_nucleotide_labels.to(device)
        original_expression_labels = original_expression_labels.to(device)
        dna_target_masks = dna_target_masks.to(device)
        rna_target_masks = rna_target_masks.to(device)

        with autocast():
            dna_logits, expression_logits = model(sequences_input)

            # --- Calculate DNA Loss ---
            dna_logits_flat = dna_logits.view(-1, NUM_NUCLEOTIDES_ORIGINAL)
            original_nucleotide_labels_flat = original_nucleotide_labels.view(-1)
            dna_target_masks_flat = dna_target_masks.view(-1)
            masked_dna_logits = dna_logits_flat[dna_target_masks_flat]
            masked_original_nucleotide_labels = original_nucleotide_labels_flat[dna_target_masks_flat]
            dna_loss_batch = torch.tensor(0.0, device=device)
            if masked_dna_logits.numel() > 0:
                dna_loss_batch = dna_criterion(masked_dna_logits, masked_original_nucleotide_labels)

            # --- Calculate RNA Loss ---
            expression_logits_flat = expression_logits.view(-1)
            original_expression_labels_flat = original_expression_labels.view(-1)
            rna_target_masks_flat = rna_target_masks.view(-1)
            masked_expression_logits = expression_logits_flat[rna_target_masks_flat]
            masked_original_expression_labels = original_expression_labels_flat[rna_target_masks_flat]
            rna_loss_batch = torch.tensor(0.0, device=device)
            if masked_expression_logits.numel() > 0:
                rna_loss_batch = rna_criterion(masked_expression_logits, masked_original_expression_labels)

            test_total_loss_sum = dna_loss_batch + rna_loss_batch
        test_total_loss += test_total_loss_sum.item()

        # --- Accumulate Metrics for Test ---
        if masked_dna_logits.numel() > 0:
            dna_predictions = torch.argmax(masked_dna_logits, dim=-1)
            test_dna_correct_predictions += (dna_predictions == masked_original_nucleotide_labels).sum().item()
            test_dna_masked_samples += masked_original_nucleotide_labels.size(0)

            # Store original and predicted nucleotides for distribution analysis
            all_original_masked_nucleotides.append(masked_original_nucleotide_labels.cpu().numpy())
            all_predicted_nucleotides.append(dna_predictions.cpu().numpy())

        if masked_expression_logits.numel() > 0:
            rna_predictions = (torch.sigmoid(masked_expression_logits) > 0.5).float()
            test_rna_correct_predictions += (rna_predictions == masked_original_expression_labels).sum().item()
            test_rna_masked_samples += masked_original_expression_labels.size(0)


avg_test_total_loss = test_total_loss / ((test_dna_masked_samples + test_rna_masked_samples) if (test_dna_masked_samples + test_rna_masked_samples) > 0 else 1)
test_dna_accuracy = test_dna_correct_predictions / (test_dna_masked_samples if test_dna_masked_samples > 0 else 1)
test_rna_accuracy = test_rna_correct_predictions / (test_rna_masked_samples if test_rna_masked_samples > 0 else 1)


print(f"\n{BOLD}--- Final Test Results ---{RESET}")
print(f"{BOLD}Test Total Loss: {GREEN}{avg_test_total_loss:.4f}{RESET}")
print(f"{BOLD}Test DNA Accuracy: {GREEN}{test_dna_accuracy:.4f}{RESET}")
print(f"{BOLD}Test RNA Accuracy: {GREEN}{test_rna_accuracy:.4f}{RESET}")

## 12. Nucleotide Distribution Analysis

This section analyzes the distribution of original and reconstructed nucleotides for the masked DNA tokens in the test set. Comparing these distributions can provide insights into how well the model is learning the underlying genomic sequences, beyond just a simple accuracy metric. A bar plot visualizes this comparison.

In [ ]:
# --- Nucleotide Distribution Analysis ---
if all_original_masked_nucleotides and all_predicted_nucleotides:
    print(f"\n{BOLD}--- Nucleotide Distribution Comparison ---{RESET}")

    # Concatenate all collected arrays
    original_nucleotides_flat = np.concatenate(all_original_masked_nucleotides)
    predicted_nucleotides_flat = np.concatenate(all_predicted_nucleotides)

    # Define nucleotide labels
    nucleotide_map = {0: 'A', 1: 'C', 2: 'G', 3: 'T', 4: 'N'}

    # Calculate distributions
    original_counts = pd.Series(original_nucleotides_flat).value_counts(normalize=True).sort_index()
    predicted_counts = pd.Series(predicted_nucleotides_flat).value_counts(normalize=True).sort_index()

    # Create a DataFrame for plotting
    distribution_df = pd.DataFrame({
        'Original': original_counts.reindex(range(NUM_NUCLEOTIDES_ORIGINAL), fill_value=0),
        'Reconstructed': predicted_counts.reindex(range(NUM_NUCLEOTIDES_ORIGINAL), fill_value=0)
    })
    distribution_df.index = [nucleotide_map[i] for i in distribution_df.index]

    print("\nOriginal Nucleotide Distribution (Masked Tokens):")
    print(distribution_df['Original'].apply(lambda x: f"{x:.4f}"))

    print("\nReconstructed Nucleotide Distribution (Predicted Tokens):")
    print(distribution_df['Reconstructed'].apply(lambda x: f"{x:.4f}"))

    # Plotting distributions
    plt.figure(figsize=(8, 6))
    distribution_df.plot(kind='bar', ax=plt.gca(), width=0.8)
    plt.title('Distribution of Original vs. Reconstructed Nucleotides (Masked Tokens)')
    plt.xlabel('Nucleotide')
    plt.ylabel('Proportion')
    plt.xticks(rotation=0)
    plt.legend()
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    plt.savefig('nucleotide_distribution_comparison.png')
    plt.show()

else:
    print("\nNo masked DNA samples were processed in the test set to compare nucleotide distributions.")

## 13. Plotting Training and Validation Metrics

This final section generates plots to visualize the training progress. It shows:
- **Total Loss**: Training and validation total loss over epochs.
- **DNA Accuracy**: Training and validation accuracy for the DNA masked prediction task.
- **RNA Accuracy**: Training and validation accuracy for the RNA masked prediction task.

These plots help in understanding model convergence, identifying potential overfitting, and assessing performance trends for each of the multi-tasks.

In [ ]:
# --- Plotting Training and Validation Metrics ---
epochs_range = range(1, NUM_EPOCHS + 1)

plt.figure(figsize=(18, 5)) # Wider figure for 3 plots

# Plot Total Loss
plt.subplot(1, 3, 1)
plt.plot(epochs_range, train_total_losses, label='Training Total Loss')
plt.plot(epochs_range, val_total_losses, label='Validation Total Loss')
plt.title('Training and Validation Total Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot DNA Accuracy
plt.subplot(1, 3, 2)
plt.plot(epochs_range, train_dna_accuracies, label='Training DNA Accuracy')
plt.plot(epochs_range, val_dna_accuracies, label='Validation DNA Accuracy')
plt.title('Training and Validation DNA Accuracy (Masked)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot RNA Accuracy
plt.subplot(1, 3, 3)
plt.plot(epochs_range, train_rna_accuracies, label='Training RNA Accuracy')
plt.plot(epochs_range, val_rna_accuracies, label='Validation RNA Accuracy (Masked)')
plt.title('Training and Validation RNA Accuracy (Masked)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('training_validation_metrics.png')
plt.show()